# Fraus - Fine-tune do BERTimbau para deteccao de ironia

Classificacao **binaria**: 0 nao-ironico, 1 ironico. A ordem esta em
`fraus.sinais.ironia` e vale aqui tambem.

**Por que uma cabeca separada** e nao mais uma classe do sinal de texto ou de
emocao: ironia nao e um sentimento, e uma relacao entre o que o texto DIZ e o
que ele SIGNIFICA. "Que atendimento maravilhoso, so esperei 3 horas" e
lexicamente positivo e pragmaticamente negativo *ao mesmo tempo*. Como quarta
classe de satisfacao, o modelo teria de escolher uma das duas leituras -- e a
informacao de que ha CONFLITO, que e o proprio sinal de ironia, se perderia.

Requer **GPU**. Este notebook e INDEPENDENTE do 01 e do 03.

## ATENCAO: o corpus nao tem download automatico

O **IDPT 2021** (tarefa de *Irony Detection in Portuguese* do IberLEF, com
15,2 mil tweets e 18,4 mil noticias anotados) **nao esta publicado para
download aberto** -- nao ha copia no GitHub nem no Hugging Face, e a pagina da
tarefa nao expoe link direto.

Antes de rodar este notebook e preciso **solicitar o corpus aos organizadores**
pela pagina da tarefa (<https://sites.google.com/inf.ufpel.edu.br/idpt2021/>) e
subir os arquivos para o Drive em:

```
/content/drive/MyDrive/fraus/dados/idpt2021/
```

A celula 2 falha com erro explicito se a pasta nao existir, e a 3 mostra as
colunas encontradas para voce ajustar os nomes -- este notebook **nunca viu os
arquivos**, entao o esquema e configuravel em vez de presumido.

Se o corpus nao for liberado a tempo, a saida honesta e **declarar a ironia
como trabalho futuro** no relatorio, e nao trocar por um corpus de sarcasmo em
ingles traduzido: seria repetir, com outro nome, o vazamento de procedencia que
ja custou o primeiro fusor.

In [ ]:
# 1. GPU conferida e Drive montado.
!pip -q install transformers datasets accelerate evaluate scikit-learn

import torch

assert torch.cuda.is_available(), (
    'Runtime sem GPU. Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU.'
)

from google.colab import drive
drive.mount('/content/drive')

import os

DIR_CORPUS = '/content/drive/MyDrive/fraus/dados/idpt2021'
DIR_CHECKPOINT = '/content/drive/MyDrive/fraus/checkpoints-ironia'
DIR_SAIDA = '/content/drive/MyDrive/fraus/modelos/bertimbau-ironia'
DIR_MODELOS = os.path.dirname(DIR_SAIDA)

if not os.path.isdir(DIR_CORPUS):
    raise FileNotFoundError(
        f'Corpus do IDPT 2021 nao encontrado em {DIR_CORPUS}. '
        'Ele NAO tem download aberto: solicite aos organizadores em '
        'https://sites.google.com/inf.ufpel.edu.br/idpt2021/ e suba os arquivos '
        'para essa pasta do Drive. Ver docs/treinamento.md, notebook 04.'
    )

os.makedirs(DIR_CHECKPOINT, exist_ok=True)
os.makedirs(DIR_SAIDA, exist_ok=True)
print('gpu:', torch.cuda.get_device_name(0))
print('arquivos no corpus:', sorted(os.listdir(DIR_CORPUS)))

## 2. O pacote `fraus`

Mesma celula dos notebooks 02 e 03: clone em `/content/fraus-src` (nunca
`/content/fraus`, que o cwd do Colab sequestraria como namespace package) e
`sys.path` ajustado a mao, porque o `.pth` do `pip install -e` so e lido no
boot do interpretador.

In [ ]:
# 2. Pacote fraus instalado a partir do repositorio.
DONO = 'joao-ped'
NOME_REPO = 'Fraus'
RAMO = 'feat/motor-e-dashboard'
DIR_REPO = '/content/fraus-src'

import shutil
import subprocess
import sys

try:
    from google.colab import userdata

    token = userdata.get('TOKEN_GITHUB')
except Exception:
    token = None

url = f'https://github.com/{DONO}/{NOME_REPO}.git'
if token:
    url = f'https://{token}@github.com/{DONO}/{NOME_REPO}.git'

shutil.rmtree(DIR_REPO, ignore_errors=True)
clone = subprocess.run(
    ['git', 'clone', '-q', '--branch', RAMO, url, DIR_REPO],
    capture_output=True, text=True,
)
if clone.returncode != 0:
    erro = clone.stderr.replace(token, '***') if token else clone.stderr
    raise RuntimeError(f'git clone falhou ({clone.returncode}): {erro.strip()}')

!pip install -q -e {DIR_REPO}

if DIR_REPO not in sys.path:
    sys.path.insert(0, DIR_REPO)

from fraus.sinais.ironia import IRONICO, NAO_IRONICO

print('classes:', {'nao_ironico': NAO_IRONICO, 'ironico': IRONICO})

## 3. Carga do corpus -- esquema CONFIGURAVEL

Este notebook nao presume o formato dos arquivos do IDPT, porque nao teve
acesso a eles. A celula abaixo:

1. le todo `.csv` e `.tsv` da pasta, detectando o separador;
2. **imprime as colunas encontradas**;
3. procura a coluna de texto e a de rotulo pelos nomes mais provaveis;
4. **para com erro nomeando as colunas disponiveis** se nao achar.

Se a deteccao falhar, ajuste `COLUNA_TEXTO` e `COLUNA_ROTULO` a mao com o que
foi impresso. Falhar apontando o que existe e melhor do que adivinhar coluna e
treinar em lixo silenciosamente.

In [ ]:
# 3. Corpus carregado da pasta do Drive.
import glob

import pandas as pd

# Ajuste se a deteccao automatica falhar -- os nomes impressos abaixo mandam.
COLUNA_TEXTO = None    # ex.: 'text', 'texto', 'tweet'
COLUNA_ROTULO = None   # ex.: 'label', 'rotulo', 'irony', 'ironia'

CANDIDATOS_TEXTO = ['text', 'texto', 'tweet', 'sentence', 'frase', 'content', 'noticia']
CANDIDATOS_ROTULO = ['label', 'rotulo', 'irony', 'ironia', 'class', 'classe', 'target', 'y']

caminhos = sorted(glob.glob(f'{DIR_CORPUS}/**/*.csv', recursive=True) +
                  glob.glob(f'{DIR_CORPUS}/**/*.tsv', recursive=True))
if not caminhos:
    raise FileNotFoundError(f'nenhum .csv ou .tsv em {DIR_CORPUS}')

partes = []
for caminho in caminhos:
    separador = '\t' if caminho.endswith('.tsv') else None  # None = sniff do pandas
    parte = pd.read_csv(caminho, sep=separador, engine='python')
    print(f'{os.path.basename(caminho):40s} {len(parte):6d} linhas  colunas={list(parte.columns)}')
    partes.append(parte)

bruto = pd.concat(partes, ignore_index=True)
colunas_minusculas = {c.lower(): c for c in bruto.columns}


def achar(candidatos, configurado, papel):
    if configurado:
        if configurado not in bruto.columns:
            raise KeyError(f'coluna {configurado!r} nao existe. Disponiveis: {list(bruto.columns)}')
        return configurado
    for candidato in candidatos:
        if candidato in colunas_minusculas:
            return colunas_minusculas[candidato]
    raise KeyError(
        f'nao achei a coluna de {papel} automaticamente. '
        f'Colunas disponiveis: {list(bruto.columns)}. '
        f'Defina COLUNA_TEXTO/COLUNA_ROTULO no topo desta celula.'
    )


coluna_texto = achar(CANDIDATOS_TEXTO, COLUNA_TEXTO, 'texto')
coluna_rotulo = achar(CANDIDATOS_ROTULO, COLUNA_ROTULO, 'rotulo')
print(f'\nusando texto={coluna_texto!r} rotulo={coluna_rotulo!r}')

dados = bruto[[coluna_texto, coluna_rotulo]].dropna()
dados.columns = ['texto', 'rotulo_bruto']
dados = dados[dados['texto'].astype(str).str.strip().astype(bool)]

# O rotulo pode vir como 0/1, "0"/"1", ou texto. Qualquer valor fora do
# mapeamento para com erro em vez de virar 0 por acidente.
MAPA_ROTULO = {
    '0': NAO_IRONICO, '1': IRONICO,
    'nao': NAO_IRONICO, 'não': NAO_IRONICO, 'no': NAO_IRONICO,
    'sim': IRONICO, 'yes': IRONICO,
    'nao-ironico': NAO_IRONICO, 'ironico': IRONICO,
    'non-irony': NAO_IRONICO, 'irony': IRONICO,
    'false': NAO_IRONICO, 'true': IRONICO,
}
chaves = dados['rotulo_bruto'].astype(str).str.strip().str.lower()
desconhecidos = sorted(set(chaves) - set(MAPA_ROTULO))
if desconhecidos:
    raise ValueError(
        f'rotulos nao reconhecidos: {desconhecidos}. '
        'Acrescente-os a MAPA_ROTULO em vez de deixar virar 0 silenciosamente.'
    )
dados['rotulo'] = chaves.map(MAPA_ROTULO).astype(int)

print(f'\n{len(dados)} exemplos')
print(dados['rotulo'].value_counts().rename({NAO_IRONICO: 'nao-ironico', IRONICO: 'ironico'}))

In [ ]:
# 4. Tokenizacao e divisao treino/teste.
from datasets import Dataset
from transformers import AutoTokenizer

MODELO_BASE = 'neuralmind/bert-base-portuguese-cased'
TAMANHO_MAXIMO = 192  # o mesmo de fraus/sinais/ironia.py

tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)

conjunto = Dataset.from_pandas(dados[['texto', 'rotulo']].reset_index(drop=True))
conjunto = conjunto.map(
    lambda lote: tokenizador(
        [str(t) for t in lote['texto']], truncation=True, max_length=TAMANHO_MAXIMO
    ),
    batched=True,
)
conjunto = conjunto.rename_column('rotulo', 'labels')
conjunto = conjunto.class_encode_column('labels')
divisao = conjunto.train_test_split(test_size=0.15, seed=42, stratify_by_column='labels')
print(divisao)

In [ ]:
# 5. Treino com pesos de classe.
import numpy as np
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

modelo = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=2)

# Corpus de ironia costuma ter muito mais exemplo nao-ironico. O peso evita que
# o modelo aprenda a responder "nao e ironia" sempre e ainda marcar boa acuracia.
contagem = np.bincount(divisao['train']['labels'], minlength=2)
pesos = len(divisao['train']) / (2 * np.maximum(contagem, 1))
pesos_tensor = torch.tensor(pesos, dtype=torch.float32).to('cuda')
print('contagem:', contagem, ' pesos:', pesos.round(3))


class TreinadorPonderado(Trainer):
    """Trainer com perda ponderada pelas duas classes."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        rotulos = inputs.pop('labels')
        saidas = model(**inputs)
        perda = nn.CrossEntropyLoss(weight=pesos_tensor)(
            saidas.logits.view(-1, 2), rotulos.view(-1)
        )
        return (perda, saidas) if return_outputs else perda


def calcular_metricas(predicao):
    preditos = np.argmax(predicao.predictions, axis=1)
    return {
        'acuracia': accuracy_score(predicao.label_ids, preditos),
        'f1_macro': f1_score(predicao.label_ids, preditos, average='macro'),
        # F1 da classe ironica isolada: e ela que o sinal usa, e a macro
        # esconderia um desempenho ruim nela atras da classe majoritaria.
        'f1_ironico': f1_score(predicao.label_ids, preditos, pos_label=IRONICO,
                               average='binary', zero_division=0),
    }


argumentos = TrainingArguments(
    output_dir=DIR_CHECKPOINT,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_ironico',  # a classe que o sinal consome
    fp16=True,
    logging_steps=100,
)

treinador = TreinadorPonderado(
    model=modelo,
    args=argumentos,
    train_dataset=divisao['train'],
    eval_dataset=divisao['test'],
    data_collator=DataCollatorWithPadding(tokenizador),
    compute_metrics=calcular_metricas,
)
treinador.train()

In [ ]:
# 6. Avaliacao e sanidade em frases de atendimento.
predicao = treinador.predict(divisao['test'])
preditos = np.argmax(predicao.predictions, axis=1)

print(classification_report(
    predicao.label_ids, preditos,
    labels=[NAO_IRONICO, IRONICO],
    target_names=['nao-ironico', 'ironico'],
    digits=4,
    zero_division=0,
))

# O IDPT e de tweets e noticias, NAO de atendimento. Estas frases nao sao
# metrica -- sao um cheiro de transferencia de dominio antes de exportar.
import torch.nn.functional as F

SONDAS = [
    ('otimo servico, so esperei 3 horas', 'esperado: ironico'),
    ('parabens pelo atendimento, nota mil, so 5 dias sem resposta', 'esperado: ironico'),
    ('qual o prazo de entrega do meu pedido', 'esperado: nao-ironico'),
    ('obrigado, resolveram rapido', 'esperado: nao-ironico'),
]
entradas = tokenizador([t for t, _ in SONDAS], return_tensors='pt',
                       truncation=True, max_length=TAMANHO_MAXIMO, padding=True).to('cuda')
modelo.eval()
with torch.inference_mode():
    probabilidades = F.softmax(modelo(**entradas).logits, dim=-1)[:, IRONICO].tolist()

print('sondas de dominio (nao sao metrica):')
for (frase, esperado), p in zip(SONDAS, probabilidades):
    print(f'  P(ironico)={p:.3f}  {frase!r}  ({esperado})')

In [ ]:
# 7. Export para o Drive.
import json

metricas = treinador.evaluate()

modelo.save_pretrained(DIR_SAIDA)
tokenizador.save_pretrained(DIR_SAIDA)

with open(f'{DIR_MODELOS}/metricas_ironia.json', 'w', encoding='utf-8') as arquivo:
    json.dump({
        'acuracia': float(metricas['eval_acuracia']),
        'f1_macro': float(metricas['eval_f1_macro']),
        'f1_ironico': float(metricas['eval_f1_ironico']),
        'classes': ['nao-ironico', 'ironico'],
        'corpus': 'IDPT 2021 (IberLEF) -- tweets e noticias',
        'limitacao': (
            'anotado em tweets e noticias, nao em atendimento de chatbot; '
            'transferencia de dominio nao verificada'
        ),
    }, arquivo, ensure_ascii=False, indent=2)

print('gravados:')
print(' ', DIR_SAIDA)
print(' ', f'{DIR_MODELOS}/metricas_ironia.json')
print()
print('Copie para o repo local:')
print(f'  {DIR_SAIDA}  ->  modelos/bertimbau-ironia/')
print(f'  {DIR_MODELOS}/metricas_ironia.json  ->  modelos/metricas_ironia.json')